In [1]:
# !pip install transformers


In [2]:
# !pip install "transformers[torch]"

In [3]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [4]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [5]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [6]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [7]:
train_data.shape

(14732, 3)

In [8]:
val_data.shape

(818, 3)

In [9]:
# Random Samplin
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
# Random Samplin
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [10]:
train_data.shape

(4000, 3)

In [11]:
val_data.shape

(500, 3)

### Data Preprocessing

In [12]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text)     # Replacing extra Line text with space
    text = re.sub(r"\s+", " ", text)     # Replacing extra spaces text with space
    text = re.sub(r"<.*?>", " ", text)     # Replacing html tag with space
    text = text.strip().lower()          # removing the extra spaces and converting all the text ito lower cases
    return text

In [13]:
# Applying the above function to clean the data 

train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = train_data["dialogue"].apply(clean_data)
val_data["summary"] = train_data["summary"].apply(clean_data)

In [14]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

### Tokenize

In [15]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [16]:
# raw data => tokenized inputs for fine tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length = 512, truncation= True)
    targets = tokenizer(data["summary"], padding="max_length", max_length = 150, truncation= True)

    inputs["labels"] = targets["input_ids"]      # token ids => add to input as label
    return inputs

In [17]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [18]:
train_dataset[0]

#     o/p:

#    1 means end of sequence, 0 means added padding
#    1. input ids => token ids
#    2. attention mask   : shows valid and padding values
#    3. labels - target => summary token

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [19]:
len(train_dataset[0]["input_ids"])

512

In [20]:
type(train_dataset)
type(val_dataset)

list

### Working with model

In [21]:
# NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [22]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")

elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device : ", device)
model.to(device)

device :  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [23]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size=8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500
    # 0  => lr default
)

In [24]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [25]:
# training the model
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.632726,0.363780
2,0.397396,0.336930
3,0.373385,0.325738
4,0.362607,0.316989
5,0.354790,0.313391
6,0.351222,0.311912


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9120209757486979, metrics={'train_runtime': 3767.4521, 'train_samples_per_second': 6.37, 'train_steps_per_second': 0.796, 'total_flos': 3248203235328000.0, 'train_loss': 0.9120209757486979, 'epoch': 6.0})

In [26]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [27]:
# model.from_pretrained("./save_summary_model")
# tokenizer.from_pretrained("./save_summary_model")

model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

### Testing the core logic for Summarization

In [28]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)    # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt"
    ).to(device)

    #generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams = 4,    
        early_stopping = True
    )

    # Decode Output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True)   # EOS, SEP
    return summary

In [29]:
test_dialogue = """
Cleo: Hey, you wanna go for coffee?
Rod: Sure, but first I have a few chores to finish up at home.
Cleo: What kind of chores. I never have to do anything except clean my room.
Rod: No chores? Get outta here.
Cleo: Seriously
Rod: I gotta vaccum, take out the garbage, mop the floor and clean my room. But that's only once a week, so it's not bad.
Cleo: I guess it's a small price to pay for still living with your folks.
Rod: Yeah, but I'm surprised your folks don't make you do chores.
Cleo: Well, I chip in every month. I pay some bills.
Rod: I have to find my own place, but I need a better job first.
Cleo: You think you'd want to work where I'm at?
Rod: I don't know. Are there any openings?
Cleo: Well, I can ask. The pay is pretty good here.
Rod: I hate earning minimum wage. 
Cleo: You'll earn enough to move out if you work at my place.
Rod: Ok, let's talk about it over coffee.
Cleo: I'll refresh my resume and you can put in a good work for me :)
Rod: Yeah, no sweat. Now get back to your chores :)
Cleo: Shut up!
Rod: Meet me at Tim Horton's at 3 if you can.
Cleo: Ok, I should be able to get out by then :)

"""

summary = summarize_dialogue(test_dialogue)

print("Summary : ", summary)

Summary :  cleo has a few chores to finish up at home. he chip in every month. he has to find his own place, but he hates earning minimum wage.
